# AIR-CIS: correspondence importance sampling against MTPFN, A-only and the exact oracle

## Description

*Written before the first run (2026-11-04). Spec and pre-registration: `research_log/entries/2026-11-04_air-cis.md`
(the spec verbatim, then a pre-run addendum with deviations D1–D7). Issue #22.*

**Question.** Is a scarce target task A predicted better by *explicit inference* of how it relates to a dense
source B, than by an *amortised* in-context learner that sees both tasks (MTPFN)? Where does each lie between the
A-only lower bound and the exact Bayes oracle, and which part of the remaining gap is the proposal's and which the
decoder's?

**The task (d = 1).** B: 128 noisy points of a random function g. A: 1–10 noisy points of
f_A(x) = h(g(T(x))), with an unknown monotone transform T of the input and an unknown output map h (identity or
affine). The learner must predict A at 32 held-out queries.

**The prior, without boundary leak.**
- g: a random tanh network with a sampled architecture (depth 1–3, width 16/32/64), mapped to [0, 1] by one
  global ECDF.
- T(x) = c + w · Φ_θ(x): a smooth monotone flow Φ_θ of complexity J ∈ {0, 2, 4, 8, 16} (p = 0.2, 0.3, 0.2, 0.15,
  0.15) with a random amplitude s_T, followed by a random window (w ∈ [0.5, 1], c ∈ [0, 1 − w]).
  *Why the window:* Φ_θ alone fixes both ends of the box, so with h = id B would reveal A's values at x = 0 and
  x = 1 exactly, and at J = 0 everywhere. A Bayes learner exploits that, and it made the predictive in notebooks
  15–16 artificially sharp at the edges. With the window, A's box maps into a random sub-interval of B's, and no
  value of A is known from B alone.
- h = id with probability 0.5, else a + b·v with b ∈ [0.3, 1], a ∈ [0, 1 − b]. Noise σ_A, σ_B ∈ [0.005, 0.03].
  Always related (no unrelated B in this stage).

**The idea: CIS (correspondence importance sampling).** The posterior over T is hard because it is
*discrete at heart*: g is not monotone, so an A point with value y_a could sit on any of the branches of g where g
≈ y_a. A smooth proposal (AIR's flow) blurs these branches together. CIS proposes them explicitly:
1. *B surrogate.* A GP on B gives μ_B(u), σ_B(u) (proposal only).
2. *Claims.* Every A point claims a set of candidate locations u in B (the 16 best-matching B points by
   N(y_a; μ_B(u), …) times a prior table π(u | x_a), plus the 4 most likely under the prior).
3. *Coherent assignments by FFBS.* Along A sorted by x, a hidden-Markov chain over the candidates with a soft
   monotone, smooth transition kernel (from the prior's stretch statistics); forward filtering, backward sampling
   draws whole assignments c, each with its exact probability q(c).
4. *A transform per assignment.* For each sampled c, and each J (and each h type), Gauss–Newton fits the transform
   to "T(x_a) lands on c_a" and to A's values through μ_B; a Laplace Gaussian around the fit is the proposal for
   θ, and the Laplace evidence gives q(J | c).
5. *Exact weights.* θ is drawn from the Laplace mixture (plus 10 % prior), and the importance weight uses the
   transported decoder D's likelihood p̂(A | B, θ), with an auxiliary term r(c | θ) that keeps the target exactly the
   posterior over (J, θ). The prediction is the weighted mixture of D's predictives.
6. *Rounds (U).* Two cheap rounds with GP-surrogate weights reshape the location prior (a KDE of where the
   weighted transforms send each A point) and add local candidates; only the last round, with decoder weights,
   enters the estimate. The proposal can be wrong without biasing the result: errors cost effective sample size.

The decoder D is AIR's transported PFN: it predicts A from A's points and B moved into A's frame by T⁻¹, with a new
feature log (T⁻¹)′(x_b) (how much the transform stretches B there), retrained on this prior.

**Setup.** Identical 256 evaluation tasks per n_A ∈ {1, 2, 3, 5, 8, 10} (CPU-drawn, fixed seed) for every method:
- **A-only**: a PFN with MTPFN's architecture trained on A alone (the clean lower bound); D's null head also shown.
- **MTPFN**: #20's architecture (1.8 M parameters) trained on this prior, A and B as two tasks.
- **CIS-U**: R = 1 and R = 3 rounds, K ∈ {32, 128} final samples.
- **IS-ref**: prior importance sampling with K = 8192 and the same decoder: what CIS would reach with a perfect
  proposal.
- **Teacher**: D at the true transform: the decoder's best case.
- **Oracle**: the exact posterior predictive given the true g and σ_A (h integrated analytically; T's latents by
  tempered SMC per J, J mixed by the SMC evidence). An upper bound no method can beat in expectation.
All trained components: seed 0 in this stage.

**What to look for, and why.**
1. *The ladder* (the main figure): NLL against n_A for A-only, MTPFN, CIS, IS-ref, teacher, oracle, and the share of
   the A-only → oracle gap each closes. Answers "where does each method lie".
2. *Paired CIS − MTPFN* per n_A with bootstrap CIs, overall and by stratum (J, number of extrema of g, true h).
   CIS should win where the posterior is multimodal (many extrema, high J, small n_A) and may lose nothing
   elsewhere.
3. *Decomposition* of CIS's gap: proposal (CIS − IS-ref) vs decoder (IS-ref − oracle). In #21 (J = 2, AIR prior)
   proposals were within 0.02 nats of IS-ref and the decoder cost 0.11–0.16 nats; if that holds here, the next
   step is the decoder, not the proposal.
4. *Proposal diagnostics*: ESS/K, branch hit rate and q-mass on the true branch combination by extrema stratum,
   R = 1 vs 3, K = 32 vs 128; the J posterior against the oracle's p(J | A, B) (calibration).
5. *Calibration of the predictive* (PIT histograms, 50/90 % interval coverage) of MTPFN, CIS and the oracle: is a
   sharp predictive earned?
6. *Leak audit*: the oracle's gain over A-only by the distance of x* to the boundary (h = id). A flat profile
   means B does not reveal A's edge values.
7. *Negative transfer*: the share of tasks where each method is worse than A-only. *Runtime* per task.

**GO / NO-GO** (this notebook's comparison; the spec's C1, C3–C6 are evaluated as written, C2/C7/C8 are out of
stage 1). **GO**: CIS-U (R = 3, K = 128) beats MTPFN by ≥ 0.05 nats with a paired 95 % CI excluding 0 at ≥ 2 of
n_A ∈ {2, 3, 5}, and is not significantly worse at any n_A. **NO-GO**: CIS is significantly better at none of
n_A ∈ {2, 3, 5}. Otherwise **INCONCLUSIVE**.

**Limits.** d = 1 only; always related (no unrelated-source stratum, so negative transfer is only tested within
the related prior); h is identity or affine (no non-monotone h); one seed per trained model; the oracle knows g
exactly (B is dense but noisy for every learner), so its gap is a bound, not a target; the decoder and MTPFN are
trained on the same prior, so prior mismatch does not enter the comparison.


## Results

*(pending: stage 1 not run)*
